### Ejercicio 1: Combinaciones Lineales y Retorno Esperado

#### Planteamiento Matemático
El rendimiento esperado de un portafolio de $n$ activos, $E[R_p]$, es una combinación lineal de los rendimientos individuales de cada activo, ponderada por su peso de asignación en la cartera:

$$E[R_p] = \sum_{i=1}^n w_i E[R_i] = w^T \cdot R$$

Donde:
* $w = [w_1, w_2, w_3]^T$ es el vector de pesos que cumple con la restricción presupuestaria de sumar 1 ($\sum_{i=1}^n w_i = 1$).
* $R = [E[R_1], E[R_2], E[R_3]]^T$ es el vector de retornos esperados de los activos.

---
- Concepto y Enfoque del Libro
 - El concepto: El cálculo se fundamenta en el producto interno estándar (o dot product) en el espacio euclidiano $\mathbb{R}^n$.
 - La trampa de NumPy: En la Sección 3.1.1, Danka nos advierte sobre un error muy común al implementar el producto punto de forma casera `(np.sum(x * y))`. Debido al fenómeno de broadcasting (donde NumPy expande automáticamente dimensiones incompatibles de forma silenciosa), un vector desalineado podría correr sin errores pero arrojar un cálculo numérico completamente incorrecto. Para evitar este comportamiento destructivo en producción, el autor recomienda utilizar estrictamente la función optimizada np.dot() o el operador @.


In [6]:
import numpy as np 
# 1. Definir los retornos esperados de 3 activos (ej: Acciones A, B, y C)
# Representan el 12%, 8% y 5% de retorno esperado respectivamente
R = np.array([0.12, 0.08, 0.05])
# 2. Definir los pesos del portafolio (deben sumar exactamente 1.0)
w = np.array([0.40, 0.30, 0.30])
# 1r condicion, la suma de los pesos debe dar exactamente 1 
# y no hay posiciones cortas 
if np.isclose(np.sum(w),1) and np.all(w>=0):
    print('portafolio valido')
    # retorno esperado del portafolio 
    E_r = R @ w
    print(f"retorno esperado: {E_r:.2%}")
# evitamos posiciones cortas 
else: 
    print('los pesos no dan 1')


portafolio valido
retorno esperado: 8.70%


### Ejercicio 2: Multiplicación de Matrices y Riesgo Marginal de los Activos

#### Planteamiento Matemático
En la optimización de portafolios, multiplicar la matriz de covarianza de retornos históricos ($\Sigma$) por el vector de pesos de asignación ($w$) mapea la estructura de variabilidad conjunta del mercado hacia la contribución de riesgo de cada activo: 

$ R_{\text{marginal}} = \Sigma \cdot w $

Donde:
* $ \Sigma \in \mathbb{R}^{3 \times 3} $ es una matriz simétrica de varianzas y covarianzas.
* $w \in \mathbb{R}^{3 \times 1}$ es el vector columna de pesos.

El vector resultante $R_{\text{marginal}} \in \mathbb{R}^{3 \times 1}$ indica cómo aporta cada activo al riesgo del portafolio global en función de su correlación con los demás componentes.

---

####  Concepto y Enfoque del Libro
* **El concepto**: En la Sección 3.2, el autor define formalmente la multiplicación matricial $A B$ como la composición de transformaciones lineales, donde cada columna de la matriz resultante es una combinación lineal de las columnas de la matriz original.
* **La dimensión en la práctica**: En la Sección 3.2.4, el texto nos enseña a resolver la diferencia entre vectores unidimensionales y vectores columna bidimensionales en NumPy. 

 **Buenas prácticas en desarrollo:**
Para evitar "abusos creativos de notación" que rompen los algoritmos en producción, debemos moldear los vectores de manera explícita como vectores columna utilizando `.reshape(-1, 1)`. Para multiplicar la matriz por el vector columna de manera óptima, el libro nos orienta a usar la función nativa `np.matmul()` o su equivalente directo en sintaxis: el operador `@`.


In [12]:
import numpy as np
# 1. Matriz de covarianza de activos (3x3)
# La diagonal representa las varianzas individuales; fuera de ella están las covarianzas
Sigma = np.array([[0.040, 0.005, 0.010],
                  [0.005, 0.025, -0.002],
                  [0.010, -0.002, 0.015]
                  ])

# 2. Vector de pesos del portafolio
w_raw = np.array([0.40, 0.30, 0.30])

# como calcularemos el riesgo marginal, usaremos una multiplicacion matricial explicita
w = w_raw.reshape(-1,1)
R_marginal = Sigma @ w
print(f"Riesgo marginal del portafolio:")
R_marginal


Riesgo marginal del portafolio:


array([[0.0205],
       [0.0089],
       [0.0079]])

### Ejercicio 3: Caso Práctico Aplicado – Varianza Total del Portafolio

#### Planteamiento Matemático
Para hallar la varianza global de un portafolio de activos ($\sigma^2_p$), se utiliza una forma cuadrática que pondera de forma simultánea los pesos en relación con la matriz de covarianzas del mercado: 

$\sigma^2_p = w^T \cdot \Sigma \cdot w$

Donde:
* $w^T \in \mathbb{R}^{1 \times 3}$ es el vector fila de pesos (la transpuesta de nuestro vector columna).
* $\Sigma \in \mathbb{R}^{3 \times 3}$ es la matriz de covarianzas.
* $w \in \mathbb{R}^{3 \times 1}$ es el vector columna de pesos. 

El resultado final de esta operación matricial consecutiva es un **valor escalar único** (un número real) que representa el riesgo consolidado del portafolio.

---

#### Concepto y Enfoque del Libro
* **El concepto**: Esta estructura matemática es un caso especial de las formas bilineales definidas en el libro como funciones de la forma $B(x, y) = x^T A y$. En las lecciones de álgebra lineal del libro, específicamente en el **Problema 9 del Capítulo 3**, se desafía al lector a implementar código en NumPy para evaluar de forma genérica formas bilineales bajo esta precisa estructura matricial.
* **La transposición**: Para ejecutar $w^T$, el libro nos enseña que la transposición simplemente consiste en "voltear" la matriz sustituyendo filas por columnas. Se accede a ella en NumPy a través de la función `np.transpose(A)` o directamente con el atributo `.T` del arreglo.
